In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sklearn as skt
import xgboost as xgb
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
import default_risk.config as cfg
import os
import xgboost as xgb
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.xgboost
import dtale
import gc

bureau_df = pd.read_parquet(cfg.CLEANS_DIR / "bureau.train-cleaned.parquet")


In [ ]:
bureau_balance_agg= pd.read_parquet(cfg.PROCESSED_DIR / "bureu_balance_agg.parquet")
bureau_df=bureau_df.merge(bureau_balance_agg,how="left",on="id_bureau")
bureau_df['has_bureau_balance_data'] = bureau_df['balance_months_balance_min'].notna().astype(int)
bureau_balance_agg.head()


In [ ]:
del bureau_balance_agg
gc.collect()

In [ ]:
bureau_agg_dic= {
    "id_curr": ["count"],

    #monetary
    "amt_credit_sum": ["max", "min","sum"],
    "amt_credit_sum_limit": ["max","mean","min"],
    "amt_annuity" : ["max","mean","min"],
    "amt_credit_sum_debt" : ["max","mean","sum"],


    #log_transformed
    "log_amt_credit_sum": ["mean","std"],

    #counters
    "cnt_credit_prolong": ["max","mean","sum"],
    "days_credit_update": ["min","max","mean"], 
    "days_credit": ["min","max","mean"], 
    "days_enddate_fact": ["max"], 

    #other
    "ratio_credit_annuity" : ["max","mean","min"],
    
    #categoricals
    "credit_active_active" : ["mean","sum"],
    "credit_active_closed" : ["mean","sum"],
    "credit_active_sold" : ["mean","sum"],
    "amt_credit_sum_overdue_is_missing": ["mean","sum"],
    "amt_credit_sum_debt_is_negative": ["mean","sum"],
    "days_enddate_fact_is_missing" : ["mean","sum"],
    "flag_have_credit_day_overdue": ["mean","sum"],
    "have_amt_credit_sum_overdue" : ["mean","sum"],
    "amt_credit_sum_limit_is_missing" : ["mean","sum"],
    "amt_credit_sum_limit_is_zero" : ["mean","sum"],
    "amt_annuity_is_missing" : ["mean","sum"],
}

In [ ]:
agg_from_bureau_balance_dict = {
    "balance_potential_on_going_loan": ["sum"], 
    
    "balance_status_score_max": ["max"], 
    
    "balance_closing_month": ["max"],
    
    "balance_months_balance_min" : ["min"]
}

In [16]:
bureau_df["log_amt_credit_sum"] = np.log1p(bureau_df["amt_credit_sum"])
bureau_df["ratio_credit_annuity"]= np.where(bureau_df["amt_annuity"] !=0, bureau_df["amt_credit_sum"] / bureau_df["amt_annuity"] , np.nan )  


bureau_df["credit_active"]=bureau_df["credit_active"].str.lower()
bureau_df= pd.get_dummies(bureau_df, columns=["credit_active"], dtype=int)

bureau_df.sort_values(["id_curr", "days_credit"],inplace=True,ascending=False)
last_two = bureau_df.groupby("id_curr").head(1)
last_two = last_two.copy()
last_two["loan_order"] = last_two.groupby("id_curr").cumcount() + 1
last_two_columns = last_two.pivot(index="id_curr", columns="loan_order")


bureau_aggregated = bureau_df.groupby("id_curr").agg(bureau_agg_dic | agg_from_bureau_balance_dict)

bureau_aggregated.columns= [f"{col[0]}_{col[1]}" for col in bureau_aggregated.columns]
bureau_aggregated= bureau_aggregated.reset_index()

last_two_columns.columns = [f"bureau_{col[0]}_loan_{col[1]}" for col in last_two_columns.columns]
last_two_columns= last_two_columns.reset_index()


last_two_columns.head()





KeyError: 'credit_active'

In [ ]:
bureau_final_df= last_two_columns.merge(bureau_aggregated,how="left",on="id_curr")
bureau_final_df.head()

In [3]:
bureau_df = pd.read_parquet(cfg.CLEANS_DIR / "bureau.train-cleaned.parquet")

bureau_balance_agg= pd.read_parquet(cfg.PROCESSED_DIR / "bureu_balance_agg.parquet")
bureau_df=bureau_df.merge(bureau_balance_agg,how="left",on="id_bureau")
bureau_df['has_bureau_balance_data'] = bureau_df['balance_months_balance_min'].notna().astype(int)
bureau_balance_agg.head()


bureau_df["ratio_credit_annuity"]= np.where(bureau_df["amt_annuity"] !=0, bureau_df["amt_credit_sum"] / bureau_df["amt_annuity"] , np.nan )  
bureau_df["completetitud_ratio"] = np.where(bureau_df["amt_credit_sum_debt"]!=0,bureau_df["amt_credit_sum"] / bureau_df["amt_credit_sum_debt"],np.nan)
bureau_df["ratio_debt_limit"] =  np.where(bureau_df["amt_credit_sum"] !=0, bureau_df["amt_credit_sum_limit"] / bureau_df["amt_credit_sum"], np.nan )  



#bureau_df["expected_vs_factical_endate"] = (bureau_df["days_credit_enddate"] * -1) - ( bureau_df["days_enddate_fact"] * -1)


bureau_df["credit_active"]=bureau_df["credit_active"].str.lower()
#bureau_df["credit_type"]=bureau_df["credit_type"].str.lower()
bureau_df= pd.get_dummies(bureau_df, columns=["credit_active"], dtype=int)
#bureau_df= pd.get_dummies(bureau_df, columns=["credit_type"], dtype=int)


bureau_df.sort_values(["id_curr", "days_credit"],inplace=True,ascending=False)
last_two = bureau_df.groupby("id_curr").head(1)
last_two = last_two.copy()
last_two["loan_order"] = last_two.groupby("id_curr").cumcount() + 1


last_two_columns = last_two.drop(columns=["id_bureau"]).pivot(index="id_curr", columns="loan_order")
last_two_columns.columns = [f"bureau_{col[0]}_loan_{col[1]}" for col in last_two_columns.columns]
last_two_columns= last_two_columns.reset_index()

bureau_df["log_amt_credit_sum"] = np.log1p(bureau_df["amt_credit_sum"])

bureau_agg_dic_active= {
    "id_curr": ["count"],

    #monetary
    "amt_credit_sum": ["max", "mean","sum","std"],
    "amt_credit_sum_limit": ["max","mean","min","std"],
    "amt_annuity" : ["max","mean","min","std"], #
    "amt_credit_sum_debt" : ["max","mean","sum","std"],


    #log_transformed
    "log_amt_credit_sum": ["mean","std"],

    #counters
    "cnt_credit_prolong": ["max","mean"], #,"sum"
    "days_credit_update": ["min","max","mean"], 
    "days_credit": ["min","max","mean"], 
    "days_credit_enddate": ["max","mean"], 

    "ratio_credit_annuity" : ["max","mean","min"],
    "completetitud_ratio" : ["mean","min"],
    "amt_annuity_is_missing" : ["mean","sum"],
    "have_amt_credit_sum_overdue" : ["mean","sum"],
    "amt_credit_max_overdue" :["max","mean","sum"],
    
}

bureau_agg_dic_closed = {
    "id_curr": ["count"],

    #monetary
    "amt_credit_sum": ["max", "mean","sum","std"],
    "amt_credit_sum_limit": ["max","mean","min"], #,"std"
    "amt_annuity" : ["max","mean","min","std"],
    "amt_credit_sum_debt" : ["max","mean","sum","std"],

    #log_transformed
    "log_amt_credit_sum": ["mean","std"],

    #counters
    #"cnt_credit_prolong": ["max","mean"], #,"sum"
    "days_credit_update": ["min","max","mean"], 
    "days_credit": ["min","max","mean"], 
    "days_enddate_fact": ["max"], 

    "ratio_credit_annuity" : ["max","mean","min"],
    "completetitud_ratio" : ["mean","min"],
    "credit_active_sold" : ["mean","sum"],
    "amt_annuity_is_missing" : ["mean","sum"],
    "amt_credit_max_overdue" :["max","mean","sum"],
    "amt_credit_max_overdue_is_missing" :["mean","sum"],
}

bureau_agg_dic= {
    "id_curr": ["count"],

    #monetary
    "amt_credit_sum": ["max", "mean","sum","std"],
    "amt_credit_sum_limit": ["max","mean","min","std"],
    "amt_annuity" : ["max","mean","min","std"],
    "amt_credit_sum_debt" : ["max","mean","sum","std"],

    #log_transformed
    "log_amt_credit_sum": ["mean","std"],

    #counters
    "cnt_credit_prolong": ["max","mean","sum"],
    "days_credit_update": ["min","max","mean"], 
    "days_credit": ["min","max","mean"], 
    "days_enddate_fact": ["max"], 
   # "expected_vs_factical_endate": ["max"],

    #other
    "ratio_credit_annuity" : ["max","mean","min"],
    "completetitud_ratio" : ["mean","min"],
    
    #categoricals
    "has_bureau_balance_data": ["sum", "mean"],
    #"credit_active_active" : ["mean","sum"],
    #"credit_active_closed" : ["mean","sum"],
    "credit_active_sold" : ["mean","sum"],
    #"amt_credit_sum_overdue_is_missing": ["mean","sum"],
    "amt_credit_sum_debt_is_negative": ["mean","sum"],
    "days_enddate_fact_is_missing" : ["mean","sum"],
    ##"flag_have_credit_day_overdue": ["mean","sum"],
    "have_amt_credit_sum_overdue" : ["mean","sum"],
    "amt_credit_sum_limit_is_missing" : ["mean","sum"],
    "amt_credit_sum_limit_is_zero" : ["mean","sum"],
    "amt_annuity_is_missing" : ["mean","sum"],
}

historical_bureau= {
    
}

agg_from_bureau_balance_dict = {
    ###"balance_potential_on_going_loan": ["sum"], 
    "balance_status_score_max": ["max"], #0.76 
    "balance_months_balance_max": ["max"], #0.76 
    "balance_months_balance_min": ["min"], #0.76 
    "balance_months_since_delincuency" : ["max"],#0.76 
    "balance_is_delincuency_sum" : ["max"],#0.76 
    "balance_is_delincuency_mean" : ["mean"],#0.76 
    ##"balance_status_0_sum" : ["sum"],
    ##"balance_is_delincuency_sum": ["sum"]
   # "balance_months_since_2_status" : ["max"],
   # "balance_months_since_3_status" : ["max"],
   # "balance_months_since_4_status" : ["max"],
   # "balance_months_since_5_status" : ["max"],    
}

active_loans= bureau_df [bureau_df["credit_active_active"] == 1]
closed_but_recent= (bureau_df["credit_active_active"] != 1) & (bureau_df["days_enddate_fact"] > -5080)
non_active_loans=  bureau_df[closed_but_recent]
bureau_active_aggregated = active_loans.groupby("id_curr").agg(bureau_agg_dic_active|agg_from_bureau_balance_dict ).add_prefix("active_")
bureau_non_active_aggregated = non_active_loans.groupby("id_curr").agg(bureau_agg_dic_closed|agg_from_bureau_balance_dict ).add_prefix("closed_")

bureau_aggregated= bureau_active_aggregated.merge(bureau_non_active_aggregated,how="outer",on="id_curr")


#
#bureau_aggregated["balance_ratio_zero_vs_delincuency"]= np.where(bureau_aggregated["balance_is_delincuency_sum"] != 0, bureau_aggregated["balance_status_0_sum"] / bureau_aggregated["balance_is_delincuency_sum"] , np.nan ) 

bureau_aggregated.columns= [f"{col[0]}_{col[1]}" for col in bureau_aggregated.columns]
bureau_aggregated= bureau_aggregated.reset_index()

#bureau_aggregated["active_closed_diff"] =bureau_aggregated["credit_active_active_sum"] - bureau_aggregated["credit_active_closed_sum"]

time_window_df= pd.read_parquet(cfg.PROCESSED_DIR / "bureau_balance_time_window.parquet")
bureau_final_df= last_two_columns.merge(bureau_aggregated,how="left",on="id_curr")
#bureau_final_df= bureau_final_df.merge(time_window_df,how="left",on="id_curr")

#bureau_final_df["balance_potential_on_going_loan_sum"] = bureau_final_df["balance_potential_on_going_loan_sum"].astype("float32")

bureau_final_df.to_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

In [ ]:
dtale.show(bureau_final_df)

In [ ]:
bureau_balance_agg= pd.read_parquet(cfg.PROCESSED_DIR / "bureu_balance_agg.parquet")
bureau_balance_agg